# 04 — IDD XML-to-YOLO Conversion

Project: Privacy-Preserving Edge AI Framework for Intelligent Transportation

Total cells: 10 (1 Markdown + 9 Code)

Objectives:
1. Load saved dataset manifests and class metadata.
2. Establish a consistent class-to-ID mapping.
3. Convert Pascal VOC XML annotations to YOLO format.
4. Save converted labels for training and validation.
5. Generate a reusable dataset.yaml configuration.

Outputs:
- configs/class_mapping.json
- configs/dataset.yaml
- Prepared YOLO label files

Important:
- The test split has no annotations and will not receive fabricated labels.
- Original images and XML annotations will remain unchanged.
- Converted labels and configurations will be reused by the training notebooks.

In [1]:
# CELL 2: Imports and paths
from pathlib import Path
import json
import xml.etree.ElementTree as ET
from collections import Counter

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")
CONFIG = PROJECT / "configs"

metadata = json.loads((CONFIG / "idd_metadata.json").read_text())
manifest_path = CONFIG / "idd_preprocessing_manifest.csv"

print("Dataset:", ROOT)
print("Project:", PROJECT)
print("Manifest:", manifest_path)

Dataset: /workspace/IDD_Detection
Project: /workspace/ITS_Project
Manifest: /workspace/ITS_Project/configs/idd_preprocessing_manifest.csv


In [2]:
# CELL 3: Load reusable manifest
import pandas as pd

manifest = pd.read_csv(manifest_path)

train_manifest = manifest[manifest["split"] == "train"].copy()
val_manifest = manifest[manifest["split"] == "val"].copy()

print("Training images:", len(train_manifest))
print("Validation images:", len(val_manifest))
print("Test images:", len(manifest[manifest["split"] == "test"]))

Training images: 31569
Validation images: 10225
Test images: 4794


In [3]:
# CELL 4: Establish consistent class IDs
classes = sorted(metadata["classes"])
class_to_id = {name: idx for idx, name in enumerate(classes)}

mapping_path = CONFIG / "class_mapping.json"
mapping_path.write_text(json.dumps(class_to_id, indent=2))

print("Classes:", len(class_to_id))
print("Mapping saved:", mapping_path)
print(class_to_id)

Classes: 15
Mapping saved: /workspace/ITS_Project/configs/class_mapping.json
{'animal': 0, 'autorickshaw': 1, 'bicycle': 2, 'bus': 3, 'car': 4, 'caravan': 5, 'motorcycle': 6, 'person': 7, 'rider': 8, 'traffic light': 9, 'traffic sign': 10, 'trailer': 11, 'train': 12, 'truck': 13, 'vehicle fallback': 14}


In [4]:
# CELL 5: Prepare YOLO label directories
YOLO_DATA = PROJECT / "prepared_dataset"
LABELS = YOLO_DATA / "labels"

for split in ["train", "val"]:
    (LABELS / split).mkdir(parents=True, exist_ok=True)

print("Prepared dataset:", YOLO_DATA)
print("Training labels:", LABELS / "train")
print("Validation labels:", LABELS / "val")

Prepared dataset: /workspace/ITS_Project/prepared_dataset
Training labels: /workspace/ITS_Project/prepared_dataset/labels/train
Validation labels: /workspace/ITS_Project/prepared_dataset/labels/val


In [17]:
# CELL 6: Collision-free XML-to-YOLO conversion function

def safe_id(entry):
    # Preserve camera and sequence identity
    return str(Path(entry).with_suffix("")).replace("/", "__")


def convert_xml_to_yolo(xml_path, output_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")

    if size is None:
        raise ValueError(f"Missing image dimensions: {xml_path}")

    width = float(size.findtext("width"))
    height = float(size.findtext("height"))

    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid image dimensions: {xml_path}")

    lines = []

    for obj in root.findall("object"):
        class_name = obj.findtext("name", default="unknown").strip()

        if class_name not in class_to_id:
            continue

        box = obj.find("bndbox")
        if box is None:
            continue

        xmin = float(box.findtext("xmin"))
        ymin = float(box.findtext("ymin"))
        xmax = float(box.findtext("xmax"))
        ymax = float(box.findtext("ymax"))

        # Clip boxes to image boundaries
        xmin = max(0.0, min(xmin, width))
        ymin = max(0.0, min(ymin, height))
        xmax = max(0.0, min(xmax, width))
        ymax = max(0.0, min(ymax, height))

        if xmax <= xmin or ymax <= ymin:
            continue

        # Normalize coordinates for YOLO
        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        box_width = (xmax - xmin) / width
        box_height = (ymax - ymin) / height

        lines.append(
            f"{class_to_id[class_name]} "
            f"{x_center:.6f} {y_center:.6f} "
            f"{box_width:.6f} {box_height:.6f}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(
        "\n".join(lines) + ("\n" if lines else "")
    )

    return len(lines)


print("Collision-free YOLO conversion function defined.")

Collision-free YOLO conversion function defined.


In [18]:
# CELL 7: Load train and validation manifests
import pandas as pd
from pathlib import Path

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")
CONFIG = PROJECT / "configs"
YOLO_DATA = PROJECT / "prepared_dataset"

manifest_path = CONFIG / "idd_preprocessing_manifest.csv"
manifest = pd.read_csv(manifest_path)

train_manifest = manifest[manifest["split"] == "train"].copy()
val_manifest = manifest[manifest["split"] == "val"].copy()

print("Training records:", len(train_manifest))
print("Validation records:", len(val_manifest))

Training records: 31569
Validation records: 10225


In [23]:

# CELL 8: Rebuild the complete dataset from original split lists
import os
import shutil
from pathlib import Path

ROOT = Path("/workspace/IDD_Detection")
PROJECT = Path("/workspace/ITS_Project")
YOLO_DATA = PROJECT / "prepared_dataset"

for split in ["train", "val"]:
    image_dir = YOLO_DATA / "images" / split
    label_dir = YOLO_DATA / "labels" / split

    # Remove previously generated files only; original dataset stays untouched
    for folder in [image_dir, label_dir]:
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)

    entries = [
        line.strip()
        for line in (ROOT / f"{split}.txt").read_text().splitlines()
        if line.strip()
    ]

    converted = 0
    skipped = []

    for entry in entries:
        image_source = (ROOT / "JPEGImages" / entry).with_suffix(".jpg")
        xml_source = (ROOT / "Annotations" / entry).with_suffix(".xml")

        if not image_source.is_file() or not xml_source.is_file():
            skipped.append(entry)
            continue

        uid = safe_id(entry)
        image_dest = image_dir / f"{uid}.jpg"
        label_dest = label_dir / f"{uid}.txt"

        os.symlink(image_source, image_dest)
        convert_xml_to_yolo(xml_source, label_dest)
        converted += 1

    print(f"{split.upper()}:")
    print(f"  Original entries: {len(entries)}")
    print(f"  Converted images: {converted}")
    print(f"  Skipped entries: {len(skipped)}")

    if skipped:
        print("  First skipped entries:", skipped[:5])


TRAIN:
  Original entries: 31569
  Converted images: 31569
  Skipped entries: 0
VAL:
  Original entries: 10225
  Converted images: 10225
  Skipped entries: 0


In [24]:
# CELL 9: Create a reusable Ultralytics dataset configuration

import yaml

dataset_config = {
    "path": str(YOLO_DATA),
    "train": "images/train",
    "val": "images/val",
    "names": {idx: name for name, idx in class_to_id.items()}
}

dataset_yaml = CONFIG / "dataset.yaml"
dataset_yaml.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True)
)

print("Dataset configuration saved:", dataset_yaml)
print(dataset_yaml.read_text())

Dataset configuration saved: /workspace/ITS_Project/configs/dataset.yaml
path: /workspace/ITS_Project/prepared_dataset
train: images/train
val: images/val
names:
  0: animal
  1: autorickshaw
  2: bicycle
  3: bus
  4: car
  5: caravan
  6: motorcycle
  7: person
  8: rider
  9: traffic light
  10: traffic sign
  11: trailer
  12: train
  13: truck
  14: vehicle fallback



In [25]:
# CELL 10: Save conversion summary for downstream notebooks

conversion_summary = {
    "dataset_root": str(ROOT),
    "prepared_dataset": str(YOLO_DATA),
    "class_mapping_path": str(mapping_path),
    "dataset_yaml": str(dataset_yaml),
    "classes": len(class_to_id),
    "train_label_files": len(conversion_counts),
    "train_objects_converted": sum(conversion_counts.values()),
    "val_label_files": len(val_conversion_counts),
    "val_objects_converted": sum(val_conversion_counts.values()),
    "test_annotations_available": False
}

summary_path = CONFIG / "yolo_conversion_summary.json"
summary_path.write_text(json.dumps(conversion_summary, indent=2))

print(json.dumps(conversion_summary, indent=2))

{
  "dataset_root": "/workspace/IDD_Detection",
  "prepared_dataset": "/workspace/ITS_Project/prepared_dataset",
  "class_mapping_path": "/workspace/ITS_Project/configs/class_mapping.json",
  "dataset_yaml": "/workspace/ITS_Project/configs/dataset.yaml",
  "classes": 15,
  "train_label_files": 4955,
  "train_objects_converted": 119003,
  "val_label_files": 3449,
  "val_objects_converted": 73438,
  "test_annotations_available": false
}


In [28]:

# CELL 11: Remove stale images and verify image-label pairing
for split in ["train", "val"]:
    image_dir = YOLO_DATA / "images" / split
    label_dir = YOLO_DATA / "labels" / split

    labels = {p.stem for p in label_dir.glob("*.txt")}
    removed = 0

    for image_path in image_dir.glob("*.jpg"):
        if image_path.stem not in labels:
            image_path.unlink()
            removed += 1

    images = {p.stem for p in image_dir.glob("*.jpg")}
    labels = {p.stem for p in label_dir.glob("*.txt")}

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Labels:", len(labels))
    print("Removed stale images:", removed)
    print("Missing labels:", len(images - labels))
    print("Orphan labels:", len(labels - images))

    assert images == labels, f"{split}: image-label mismatch"

print("\nSUCCESS: Image-label pairing verified.")



TRAIN
Images: 31569
Labels: 31569
Removed stale images: 4955
Missing labels: 0
Orphan labels: 0

VAL
Images: 10225
Labels: 10225
Removed stale images: 3449
Missing labels: 0
Orphan labels: 0

SUCCESS: Image-label pairing verified.


In [30]:

# CELL 12: Reload best model, evaluate, and save summary
from pathlib import Path
import json
from ultralytics import YOLO

PROJECT = Path("/workspace/ITS_Project")
ROOT = Path("/workspace/IDD_Detection")
RESULT_DIR = PROJECT / "results" / "training" / "yolov8n"

# Load the best checkpoint from the full-dataset run
RUN_BEST = RESULT_DIR / "idd_yolov8n_full" / "weights" / "best.pt"

if not RUN_BEST.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {RUN_BEST}")

model = YOLO(str(RUN_BEST))

# Recompute validation metrics on the complete validation split
DATA_YAML = PROJECT / "configs" / "dataset.yaml"

metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project=str(RESULT_DIR),
    name="validation_full_recheck",
    exist_ok=True
)

summary = {
    "model": "YOLOv8n",
    "training_images": 31569,
    "validation_images": 10225,
    "epochs_requested": 20,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50_95": float(metrics.box.map),
    "best_checkpoint": str(RUN_BEST),
    "evaluation_split": "validation"
}

summary_path = RESULT_DIR / "full_dataset_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("\nSummary saved to:", summary_path)


Ultralytics 8.4.177 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA H100 80GB HBM3 MIG 3g.40gb, 40320MiB)
Model summary (fused): 73 layers, 3,136,109 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1541.9±320.6 MB/s, size: 475.1 KB)
val: Scanning /workspace/ITS_Project/prepared_dataset/labels/val.cache... 10225 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10225/10225 2.9Git/s 0.0s
val: /workspace/ITS_Project/prepared_dataset/images/val/highquality_16k__HYD-2018-08-24_13-02-50__0004247.jpg: 1 duplicate labels removed
val: /workspace/ITS_Project/prepared_dataset/images/val/highquality_16k__HYD-2018-08-24_13-22-50__0007776.jpg: 1 duplicate labels removed
val: /workspace/ITS_Project/prepared_dataset/images/val/highquality_16k__HYD-2018-08-24_13-22-50__0008384.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 640/640 26.1it/s 24.5s0.0ss
                   